# Min-Sliced Lifted Transport Plan

A one-dimensional projection can define a feasible high-dimensional coupling: sort the samples after projection and lift this sorted matching back to the ambient plane.  This notebook compares such a lifted min-sliced plan with the classical quadratic $W_2$ matching.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    box_axes,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

from matplotlib.colors import to_rgb
import ot

NAME = "min-sliced-transport-plan"
out = figure_dir(NAME)
rng = np.random.default_rng(52)

The selected direction is found by a tiny deterministic sweep over angles.  This is only a visual construction of a feasible plan, not an algorithmic benchmark.

In [2]:
def rot(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

n = 30
x = np.vstack([
    rng.normal(size=(15, 2)) @ np.diag([0.32, 0.11]) @ rot(np.deg2rad(20)).T + np.array([-0.62, -0.18]),
    rng.normal(size=(15, 2)) @ np.diag([0.26, 0.12]) @ rot(np.deg2rad(-12)).T + np.array([-0.35, 0.54]),
])
y = np.vstack([
    rng.normal(size=(12, 2)) @ np.diag([0.24, 0.11]) @ rot(np.deg2rad(-25)).T + np.array([0.62, -0.48]),
    rng.normal(size=(18, 2)) @ np.diag([0.28, 0.13]) @ rot(np.deg2rad(30)).T + np.array([0.86, 0.36]),
])

def lifted_pairs(theta):
    px, py = x @ theta, y @ theta
    ix, iy = np.argsort(px), np.argsort(py)
    pairs = [(int(ix[k]), int(iy[k]), 1.0/n) for k in range(n)]
    cost = sum(np.sum((x[i]-y[j])**2) for i, j, _ in pairs) / n
    return cost, pairs

angles = np.linspace(0, np.pi, 91, endpoint=False)
dirs = np.column_stack([np.cos(angles), np.sin(angles)])
costs = np.array([lifted_pairs(theta)[0] for theta in dirs])
best = int(np.argmin(costs))
theta = dirs[best]
_, sliced_pairs = lifted_pairs(theta)
a = np.ones(n) / n
P_w2 = ot.emd(a, a, ot.dist(x, y))
w2_pairs = [(i, j, float(P_w2[i, j])) for i in range(n) for j in range(n) if P_w2[i, j] > 1e-10]
pts = np.vstack([x, y])
xlim, ylim = padded_limits(pts, pad=0.14)
print(float(angles[best]), float(costs[best]))

0.724982920059183 1.4792185109116809


In [3]:
def decorate(ax):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    remove_axes(ax)

fig, ax = plt.subplots(figsize=(2.30, 2.18))
draw_point_clouds(ax, x, y, base_size=DIRAC_MARKER_SIZE * 0.78)
line = np.vstack([-1.60*theta, 1.60*theta])
ax.plot(line[:,0], line[:,1], color=ORANGE, lw=1.20, alpha=0.88, zorder=0)
ax.arrow(1.18*theta[0], 1.18*theta[1], 0.28*theta[0], 0.28*theta[1], width=0.009, head_width=0.070, head_length=0.095, length_includes_head=True, color=ORANGE, alpha=0.88, zorder=0)
decorate(ax)
save_pdf(fig, out / "direction.pdf", pad_inches=0.055)
plt.close(fig)

for pairs, filename, color in [(sliced_pairs, "lifted-plan.pdf", ORANGE), (w2_pairs, "w2-plan.pdf", VIOLET)]:
    fig, ax = plt.subplots(figsize=(2.30, 2.18))
    draw_transport_segments(ax, x, y, pairs, color=color, min_width=0.18, max_width=0.88, alpha_scale=0.50, zorder=1)
    draw_point_clouds(ax, x, y, base_size=DIRAC_MARKER_SIZE * 0.78)
    decorate(ax)
    save_pdf(fig, out / filename, pad_inches=0.055)
    plt.close(fig)